In [21]:

import requests
import json
import time
import pathlib
import urllib.parse
import os
import re
import pandas as pd
from dotenv import load_dotenv

# Load API credentials
load_dotenv(override=True)

APP_ID = os.getenv("APP_ID")
API_KEY = os.getenv("API_KEY")

INDEX = "Listing_sold_production"
URL = f"https://{APP_ID.lower()}-dsn.algolia.net/1/indexes/*/queries"

QS = {
    "x-algolia-agent": "Algolia for JavaScript (4.14.3); Browser"
}

headers = {
    "x-algolia-application-id": APP_ID,
    "x-algolia-api-key": API_KEY,
    "Content-Type": "application/x-www-form-urlencoded",
    "Origin": "https://www.grailed.com",
    "Referer": "https://www.grailed.com/",
    "User-Agent": "Mozilla/5.0 (academic project)",
}

# Paths
BASE_DIR = pathlib.Path("..")  # Project root

RAW_DIR = BASE_DIR / "extraction" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "grailed"
QUERIES_FILE = BASE_DIR / "data" / "brand_list.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FOOTWEAR_CATEGORIES = [
    "womens_footwear.boots",
    "womens_footwear.heels",
    "womens_footwear.platforms",
    "womens_footwear.mules",
    "womens_footwear.flats",
    "womens_footwear.hitop_sneakers",
    "womens_footwear.lowtop_sneakers",
    "womens_footwear.sandals",
    "womens_footwear.slip_ons",
    "footwear.boots",
    "footwear.leather",
    "footwear.formal_shoes",
    "footwear.hitop_sneakers",
    "footwear.lowtop_sneakers",
    "footwear.sandals",
    "footwear.slip_ons",
]

FOOTWEAR_FILTER = "(" + " OR ".join(
    f"category_path:{category}"
    for category in FOOTWEAR_CATEGORIES
) + ")"


def fetch(query, page, lo=0, hi=1000000):

    p = {
        "query": query,
        "hitsPerPage": 40,
        "page": page,
        "numericFilters": json.dumps([
            f"sold_price>={lo}",
            f"sold_price<={hi}"
        ]),
        "facetFilters": json.dumps([
            [f"category_path:{category}" for category in FOOTWEAR_CATEGORIES]
        ]),
    }

    body = {
        "requests": [{
            "indexName": INDEX,
            "params": urllib.parse.urlencode(p)
        }]
    }

    r = requests.post(
        URL,
        params=QS,
        headers=headers,
        data=json.dumps(body),
        timeout=(15, 90)
    )

    if r.status_code != 200:
        raise RuntimeError(f"{r.status_code}: {r.text[:300]}")

    return r.json()["results"][0]


def crawl(query, lo, hi, output_dir, depth=0):

    first = fetch(query, 0, lo, hi)

    n = first["nbHits"]
    indent = "  " * depth

    if n == 0:
        return

    # Split price ranges if there are too many results
    if n > 1000 and hi > lo:

        mid = (lo + hi) // 2

        crawl(query, lo, mid, output_dir, depth + 1)
        crawl(query, mid + 1, hi, output_dir, depth + 1)

        return

    pages = min(first["nbPages"], 25)

    for page in range(pages):

        res = first if page == 0 else fetch(query, page, lo, hi)

        name = f"{lo}-{hi}_p{page}.json"

        (output_dir / name).write_text(
            json.dumps(res),
            encoding="utf-8"
        )

        print(f"{indent}  Saved page {page + 1}/{pages}")

        if page < pages - 1:
            time.sleep(1)


def run_scraper():
    queries_df = pd.read_csv(QUERIES_FILE)

    queries = queries_df["query"].dropna().astype(str).str.strip()

    for query in queries:

        if not query:
            continue

        # Create a safe folder name
        folder_name = re.sub(r'[<>:"/\\|?*]', "_", query)
        output_dir = RAW_DIR / folder_name

        output_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n{'=' * 50}")
        print(f"Starting query: {query}")
        print(f"Output folder: {output_dir}")
        print(f"{'=' * 50}")

        crawl(
            query=query,
            lo=0,
            hi=5000,
            output_dir=output_dir
        )

        print(f"Finished query: {query}")


def extract_brand_data(brand_dir):

    rows = []

    for f in brand_dir.glob("*.json"):

        with open(f, encoding="utf-8") as file:
            data = json.load(file)

        for h in data["hits"]:

            drops = h.get("price_drops") or []

            listed_at = pd.to_datetime(
                h.get("created_at"), errors="coerce", utc=True
            )

            sold_at = pd.to_datetime(
                h.get("sold_at"), errors="coerce", utc=True
            )

            days_on_market = (
                (sold_at - listed_at).total_seconds() / 86400
                if pd.notna(listed_at) and pd.notna(sold_at)
                else None
            )

            rows.append({
                "listing_id": h["id"],
                "brand": h.get("designer_names"),
                "title": h.get("title"),
                "category": h.get("category"),
                "subcategory": h.get("category_path"),
                "size": h.get("size"),
                "condition": h.get("condition"),
                "color": h.get("color"),
                "country": h.get("location"),
                "sold_price": h.get("sold_price"),
                "last_asking_price": h.get("price_i"),
                "first_price": drops[0] if drops else h.get("price_i"),
                "n_price_drops": len(drops),
                "followers": h.get("followerno"),
                "listed_at": listed_at,
                "sold_at": sold_at,
                "days_on_market": days_on_market,
            })

    df = pd.DataFrame(rows)

    if df.empty:
        print(f"No data found for {brand_dir.name}")
        return

    # Clean data
    df = df.drop_duplicates("listing_id")
    df["size"] = df["size"].astype("string")
    df = df[df["sold_price"] > 0]

    output_file = PROCESSED_DIR / f"{brand_dir.name}.csv"
    df.to_csv(output_file, index=False)

    print(f"{brand_dir.name}: {df.shape[0]} listings saved to {output_file}")


In [22]:
run_scraper()


Starting query: nike
Output folder: ..\extraction\raw\nike
                  Saved page 1/23
                  Saved page 2/23
                  Saved page 3/23
                  Saved page 4/23
                  Saved page 5/23
                  Saved page 6/23
                  Saved page 7/23
                  Saved page 8/23
                  Saved page 9/23
                  Saved page 10/23
                  Saved page 11/23
                  Saved page 12/23
                  Saved page 13/23
                  Saved page 14/23
                  Saved page 15/23
                  Saved page 16/23
                  Saved page 17/23
                  Saved page 18/23
                  Saved page 19/23
                  Saved page 20/23
                  Saved page 21/23
                  Saved page 22/23
                  Saved page 23/23
                      Saved page 1/19
                      Saved page 2/19
                      Saved page 3/19
                      Saved pa

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [12]:
for brand_dir in RAW_DIR.iterdir():

    if brand_dir.is_dir():
        extract_brand_data(brand_dir)


jordan 1: 25884 listings saved to ..\data\grailed\jordan 1.csv
No data found for nike
No data found for yeezy


In [14]:
import base64

decoded = base64.b64decode(API_KEY).decode("utf-8")

print(decoded[64:])

validUntil=1794935896&userToken=19464411&filters=visible_by%3A+19464411


In [18]:
fetch("nike", 0, 0, 5000)

{'hits': [],
 'nbHits': 0,
 'page': 0,
 'nbPages': 0,
 'hitsPerPage': 40,
 'exhaustiveNbHits': False,
 'exhaustiveTypo': False,
 'exhaustive': {'nbHits': False, 'typo': False},
 'query': 'nike',
 'params': 'query=nike&hitsPerPage=40&page=0&numericFilters=%5B%22sold_price%3E%3D0%22%2C+%22sold_price%3C%3D5000%22%5D&filters=%28category_path%3Awomens_footwear.boots+OR+category_path%3Awomens_footwear.heels+OR+category_path%3Awomens_footwear.platforms+OR+category_path%3Awomens_footwear.mules+OR+category_path%3Awomens_footwear.flats+OR+category_path%3Awomens_footwear.hitop_sneakers+OR+category_path%3Awomens_footwear.lowtop_sneakers+OR+category_path%3Awomens_footwear.sandals+OR+category_path%3Awomens_footwear.slip_ons+OR+category_path%3Afootwear.boots+OR+category_path%3Afootwear.leather+OR+category_path%3Afootwear.formal_shoes+OR+category_path%3Afootwear.hitop_sneakers+OR+category_path%3Afootwear.lowtop_sneakers+OR+category_path%3Afootwear.sandals+OR+category_path%3Afootwear.slip_ons%29&validU